# Class 8. More exam prep

## Plan:

<h2>

1) Pandas & SQL practice <br>

</h2>

## Imports

In [ ]:
import pandas as pd
import psycopg2

from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT # to create databases
from pandas.testing import assert_frame_equal

from typing import Iterable, Callable

## Global variables

In [ ]:
db_nm = 'postgres'
usr_nm = 'postgres'
host = 'localhost'
port = '5432'

#IMPORTANT! DO NOT ENTER THE PASSWORD HERE, SAVE IT IN THE FILE AND READ IT FROM THE FILE

path_to_pwd_file = 'pwd.txt'

with open(path_to_pwd_file, 'r') as file:
    pwd = file.read().strip('\n')

## Functions that we will need later

In [ ]:
def connect(pwd: str,
            usr_nm: str = 'postgres', 
            host: str = 'localhost',
            port: str = '5432',
            db_nm: str = 'postgres'
           ) -> psycopg2.extensions.connection | None:

    '''
    Connect to a db

    Inputs:
        usr_nm: str - username for the database
        host: str - host of the sql cluster/ DB
        port: str - target port
        db_nm: str - name of the database to connect to
        pwd: str - user password

    Outputs:
        psycopg2.extensions.connection | None - connection to the database
    '''

    conn = None
    
    try:
        conn = psycopg2.connect(
            host=host,
            database=db_nm,
            user=usr_nm,
            port=port,
            password=pwd
        )

    except:
        print('Connection failure!')

    else:
        print('Connection success!')

    finally:
        return conn


def create_db(new_db_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    '''
    Create a new db. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection (maybe with cursor)
        is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
        conn: psycopg2.extensions.connection - connection to a database
        cur: psycopg2.extensions.cursor - cursor in a database
        new_db_nm: str - the name of the database to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"

    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

        cur = conn.cursor()

    try:
        cur.execute(f'CREATE DATABASE {new_db_nm};')
    
    except:
        print(f"Couldn't create database. The database {new_db_nm} probably already exists")

    else:
        print(f'Database {new_db_nm} created!')

    finally:
        if conn_flg:
            return conn, cur
        else:
            return cur

def del_db(db_to_del_nm: str,
           conn: psycopg2.extensions.connection | None = None,
           cur: psycopg2.extensions.cursor | None = None
          ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    '''
    Delete a target database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection
        (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       db_to_del_nm: str - the name of the database to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''
    
    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

        cur = conn.cursor()

    try:
        cur.execute(f'DROP DATABASE {db_to_del_nm};')
    
    except:
        print(f"Couldn't drop database. The database {db_to_del_nm} probably already exists")

    else:
        print(f'Database {db_to_del_nm} dropped!')

    finally:
        if conn_flg:
            return conn, cur
        else:
            return cur

def create_table(table_nm: str,
                 table_cols: list[str],
                 table_dtypes: list[str],
                 conn: psycopg2.extensions.connection | None = None,
                 cur: psycopg2.extensions.cursor | None = None 
                ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:
    '''
    Create table in a database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection
        (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       table_nm: str - the name of the table to be created

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        cur = conn.cursor()

    
    pairs = []

    for col, dtype in zip(table_cols, table_dtypes):
        pairs.append(' '.join([col, dtype])+',')

    pairs[-1] = pairs[-1].strip(',')

    cols = '\n'.join(pairs)

    # print(cols)
    
    try:
        cur.execute(f'''CREATE TABLE {table_nm} (
                id SERIAL,
                {cols}
                    );''')

        conn.commit()

    except Exception as e:
        print('Failed to create table, such table already exists')
        print(str(e))

    else:
        print(f'Succesfully created table {table_nm}')

    finally:

        if conn_flg:
                return conn, cur
        else:
            return cur

def del_table(table_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:
    '''
    Create table in a database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur]. If only connection
        (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       table_nm: str - the name of the table to be deleted

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        cur = conn.cursor()

    try:
        cur.execute(f'''DROP TABLE {table_nm};''')
        conn.commit()

    except Exception as e:
        print('Failed to drop table, it probably does not exist')
        # print(str(e))

    else:
        print(f'Succesfully dropped table {table_nm}')

    finally:

        if conn_flg:
                return conn, cur
        else:
            return cur

def upload_to_table(table_nm: str,
                    df: pd.DataFrame,
                    conn: psycopg2.extensions.connection | None = None,
                    cur: psycopg2.extensions.cursor | None = None
                   ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:
    '''
    Upload the pd.DataFrame into the table in the database. For the argument AT LEAST ONE of the following variables should be specified: [conn, cur].
        If only connection (maybe with cursor) is passed returns a tuple of the connection and cursor objects if only cursor is passed returns a cursor
        object

    Inputs:
       conn: psycopg2.extensions.connection - connection to a database
       cur: psycopg2.extensions.cursor - cursor in a database
       table_nm: str - the name of the table to be deleted
       df: pd.DataFrame - the table to upload

    Outputs:
        (psycopg2.extensions.connection, psycopg2.extensions.cursor) | psycopg2.extensions.cursor - used objects

    '''
    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"
    
    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False
    
    if conn is not None:
        cur = conn.cursor()

    try:
    
        for idx, row_data in df.iterrows():
            cur.execute(f'INSERT INTO {table_nm} ({', '.join(df.columns)}) VALUES ({('%s,'*len(df.columns)).strip(',')})', tuple(row_data))

    except Exception as e:
        print('Failed to append data to SQL table!')
        print(str(e))
        
    else:
        print(f'Added data to table {table_nm}')
    
    finally:
        if conn_flg:
                return conn, cur
        else:
            return cur

## Exercise 1

Table: `Transactions`

| Column Name | Type    |
|-------------|---------|
| id          | int     |
| country     | varchar |
| state       | enum    |
| amount      | int     |
| trans_date  | date    |

id is the primary key of this table.  
The table has information about incoming transactions.  
The state column is an enum of type ["approved", "declined"].

Write a solution to find for each month and country, the number of transactions and their total amount, the number of approved transactions and their total amount.

Return the result table in any order.

The result format is in the following example.

Example 1:

Input:  
Transactions table:

| id   | country | state    | amount | trans_date |
|------|---------|----------|--------|------------|
| 121  | US      | approved | 1000   | 2018-12-18 |
| 122  | US      | declined | 2000   | 2018-12-19 |
| 123  | US      | approved | 2000   | 2019-01-01 |
| 124  | DE      | approved | 2000   | 2019-01-07 |

Output: 

| month   | country | trans_count | approved_count | trans_total_amount | approved_total_amount |
|---------|---------|-------------|----------------|--------------------|-----------------------|
| 2018-12 | US      | 2           | 1              | 3000               | 1000                  |
| 2019-01 | US      | 1           | 1              | 2000               | 2000                  |
| 2019-01 | DE      | 1           | 1              | 2000               | 2000                  |


In [ ]:
def sol_1(transactions: pd.DataFrame) -> pd.DataFrame:
    ###your code here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    if "month" in df.columns and "country" in df.columns:
        return df.sort_values(["month", "country"]).reset_index(drop=True)
    return df.reset_index(drop=True)

def _case(rows, expected_rows):
    tx = pd.DataFrame(rows, columns=["id", "country", "state", "amount", "trans_date"])
    tx["trans_date"] = pd.to_datetime(tx["trans_date"])
    got = sol_1(tx)
    exp = pd.DataFrame(
        expected_rows,
        columns=["month", "country", "trans_count", "approved_count", "trans_total_amount", "approved_total_amount"],
    )
    assert_frame_equal(got, exp, check_dtype=False)

def run():
    cases = [
        ("example", lambda: _case(
            [(121, "US", "approved", 1000, "2018-12-18"),
             (122, "US", "declined", 2000, "2018-12-19"),
             (123, "US", "approved", 2000, "2019-01-01"),
             (124, "DE", "approved", 2000, "2019-01-07")],
            [("2018-12", "US", 2, 1, 3000, 1000),
             ("2019-01", "DE", 1, 1, 2000, 2000),
             ("2019-01", "US", 1, 1, 2000, 2000)],
        )),
        ("single_approved", lambda: _case(
            [(1, "FR", "approved", 500, "2020-05-15")],
            [("2020-05", "FR", 1, 1, 500, 500)],
        )),
        ("single_declined", lambda: _case(
            [(1, "FR", "declined", 500, "2020-05-15")],
            [("2020-05", "FR", 1, 0, 500, 0)],
        )),
        ("multiple_months_same_country", lambda: _case(
            [(1, "US", "approved", 100, "2020-01-10"),
             (2, "US", "approved", 200, "2020-01-15"),
             (3, "US", "declined", 50, "2020-02-01")],
            [("2020-01", "US", 2, 2, 300, 300),
             ("2020-02", "US", 1, 0, 50, 0)],
        )),
        ("empty_input", lambda: _case([], [])),
    ]
    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(f'Failed: {failed}')
    
run()

## Exercise 2

Table: `Products`

| Column Name  | Type    |
|--------------|---------|
| product_id   | int     |
| product_name | varchar |
| description  | varchar |

(product_id) is the unique key for this table.  
Each row in the table represents a product with its unique ID, name, and description.

Write a solution to find all products whose description contains a valid serial number pattern. A valid serial number follows these rules:

- It starts with the letters `SN` (case-sensitive).
- Followed by exactly 4 digits.
- It must have a hyphen (`-`) followed by exactly 4 digits.
- The serial number must be within the description (it may not necessarily start at the beginning).

Return the result table ordered by `product_id` in ascending order.

The result format is in the following example.

Example:

Input:  
Products table:

| product_id | product_name | description                                          |
|------------|--------------|------------------------------------------------------|
| 1          | Widget A     | This is a sample product with SN1234-5678            |
| 2          | Widget B     | A product with serial SN9876-1234 in the description |
| 3          | Widget C     | Product SN1234-56789 is available now                |
| 4          | Widget D     | No serial number here                                |
| 5          | Widget E     | Check out SN4321-8765 in this description            |

Output:

| product_id | product_name | description                                          |
|------------|--------------|------------------------------------------------------|
| 1          | Widget A     | This is a sample product with SN1234-5678            |
| 2          | Widget B     | A product with serial SN9876-1234 in the description |
| 5          | Widget E     | Check out SN4321-8765 in this description            |

In [ ]:
def sol_2(products: pd.DataFrame) -> pd.DataFrame:
    ###your code here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values("product_id").reset_index(drop=True)

def _case(rows, expected_rows):
    products = pd.DataFrame(rows, columns=["product_id", "product_name", "description"])
    got = sol_2(products)
    exp = pd.DataFrame(expected_rows, columns=["product_id", "product_name", "description"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)

def run():
    cases = [
        ("example", lambda: _case(
            [
                (1, "Widget A", "This is a sample product with SN1234-5678"),
                (2, "Widget B", "A product with serial SN9876-1234 in the description"),
                (3, "Widget C", "Product SN1234-56789 is available now"),
                (4, "Widget D", "No serial number here"),
                (5, "Widget E", "Check out SN4321-8765 in this description"),
            ],
            [
                (1, "Widget A", "This is a sample product with SN1234-5678"),
                (2, "Widget B", "A product with serial SN9876-1234 in the description"),
                (3, "Widget C", "Product SN1234-56789 is available now"),
                (5, "Widget E", "Check out SN4321-8765 in this description"),
            ],
        )),
        ("multiple_matches", lambda: _case(
            [(1, "X", "SN1234-5678 and SN8765-4321")],
            [(1, "X", "SN1234-5678 and SN8765-4321")],
        )),
        ("no_match", lambda: _case(
            [(1, "X", "SN12345-6789"), (2, "Y", "sn1234-5678")],
            [],
        )),
        ("boundary", lambda: _case(
            [(1, "A", "SN0000-0000")],
            [(1, "A", "SN0000-0000")],
        )),
        ("with_extra_text", lambda: _case(
            [(1, "B", "abc SN9999-9999 def")],
            [(1, "B", "abc SN9999-9999 def")],
        )),
    ]
    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(f'Failed: {failed}')
    
run()

## Exercise 3

Table: `Activity`

| Column Name   | Type    |
|---------------|---------|
| machine_id    | int     |
| process_id    | int     |
| activity_type | enum    |
| timestamp     | float   |

(machine_id, process_id, activity_type) is the primary key of this table.  
machine_id is the ID of a machine.  
process_id is the ID of a process running on the machine with ID machine_id.  
activity_type is an ENUM of type ('start', 'end').  
timestamp is a float representing the current time in seconds.  
'start' means the machine starts the process at the given timestamp and 'end' means the machine ends the process at the given timestamp.  
The 'start' timestamp will always be before the 'end' timestamp for every (machine_id, process_id) pair.  
It is guaranteed that each (machine_id, process_id) pair has a 'start' and 'end' timestamp.

There is a factory website that has several machines each running the same number of processes. Write a solution to find the average time each machine takes to complete a process.

The time to complete a process is the 'end' timestamp minus the 'start' timestamp. The average time is calculated by the total time to complete every process on the machine divided by the number of processes that were run.

The resulting table should have the machine_id along with the average time as `processing_time`, which should be rounded to 3 decimal places.

Return the result table in any order.

The result format is in the following example.

Example 1:

Input:  
Activity table:

| machine_id | process_id | activity_type | timestamp |
|------------|------------|---------------|-----------|
| 0          | 0          | start         | 0.712     |
| 0          | 0          | end           | 1.520     |
| 0          | 1          | start         | 3.140     |
| 0          | 1          | end           | 4.120     |
| 1          | 0          | start         | 0.550     |
| 1          | 0          | end           | 1.550     |
| 1          | 1          | start         | 0.430     |
| 1          | 1          | end           | 1.420     |
| 2          | 0          | start         | 4.100     |
| 2          | 0          | end           | 4.512     |
| 2          | 1          | start         | 2.500     |
| 2          | 1          | end           | 5.000     |

Output: 

| machine_id | processing_time |
|------------|-----------------|
| 0          | 0.894           |
| 1          | 0.995           |
| 2          | 1.456           |

In [ ]:
def sol_3(activity: pd.DataFrame) -> pd.DataFrame:
    ###your code here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values("machine_id").reset_index(drop=True)

def _case(rows, expected_rows):
    activity = pd.DataFrame(rows, columns=["machine_id", "process_id", "activity_type", "timestamp"])
    got = sol_3(activity)
    exp = pd.DataFrame(expected_rows, columns=["machine_id", "processing_time"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False, atol=1e-3)

def run():
    cases = [
        ("example", lambda: _case(
            [
                (0, 0, "start", 0.712),
                (0, 0, "end", 1.520),
                (0, 1, "start", 3.140),
                (0, 1, "end", 4.120),
                (1, 0, "start", 0.550),
                (1, 0, "end", 1.550),
                (1, 1, "start", 0.430),
                (1, 1, "end", 1.420),
                (2, 0, "start", 4.100),
                (2, 0, "end", 4.512),
                (2, 1, "start", 2.500),
                (2, 1, "end", 5.000),
            ],
            [
                (0, 0.894),
                (1, 0.995),
                (2, 1.456),
            ],
        )),
        ("single_machine_single_process", lambda: _case(
            [(0, 0, "start", 1.0), (0, 0, "end", 3.0)],
            [(0, 2.000)],
        )),
        ("multiple_machines", lambda: _case(
            [(0, 0, "start", 0.0), (0, 0, "end", 1.0),
             (1, 0, "start", 0.0), (1, 0, "end", 2.0),
             (1, 1, "start", 2.0), (1, 1, "end", 5.0)],
            [(0, 1.000), (1, 2.500)],
        )),
        ("rounding", lambda: _case(
            [(0, 0, "start", 1.0005), (0, 0, "end", 2.0005)],
            [(0, 1.000)],
        )),
        ("negative_timestamps", lambda: _case(
            [(0, 0, "start", -5.0), (0, 0, "end", -3.0)],
            [(0, 2.000)],
        )),
    ]
    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(f'Failed: {failed}')
    
run()

## Exercise 4

Table: `Employees`

| Column Name | Type    |
|-------------|---------|
| employee_id | int     |
| name        | varchar |
| reports_to  | int     |
| age         | int     |

employee_id is the column with unique values for this table.  
This table contains information about the employees and the id of the manager they report to. Some employees do not report to anyone (reports_to is null). 

For this problem, we will consider a manager an employee who has at least 1 other employee reporting to them.

Write a solution to report the ids and the names of all managers, the number of employees who report directly to them, and the average age of the reports rounded to the nearest integer (round down).

Return the result table ordered by employee_id.

The result format is in the following example.

Example 1:

Input:  
Employees table:

| employee_id | name    | reports_to | age |
|-------------|---------|------------|-----|
| 9           | Hercy   | null       | 43  |
| 6           | Alice   | 9          | 41  |
| 4           | Bob     | 9          | 36  |
| 2           | Winston | null       | 37  |

Output: 

| employee_id | name  | reports_count | average_age |
|-------------|-------|---------------|-------------|
| 9           | Hercy | 2             | 39          |

Example 2:

Input:  
Employees table:

| employee_id | name    | reports_to | age |
|-------------|---------|------------|-----|
| 1           | Michael | null       | 45  |
| 2           | Alice   | 1          | 38  |
| 3           | Bob     | 1          | 42  |
| 4           | Charlie | 2          | 34  |
| 5           | David   | 2          | 40  |
| 6           | Eve     | 3          | 37  |
| 7           | Frank   | null       | 50  |
| 8           | Grace   | null       | 48  |

Output: 

| employee_id | name    | reports_count | average_age |
|-------------|---------|---------------|-------------|
| 1           | Michael | 2             | 40          |
| 2           | Alice   | 2             | 37          |
| 3           | Bob     | 1             | 37          |

In [ ]:
def sol_4(employees: pd.DataFrame) -> pd.DataFrame:
    ###your code here

In [ ]:
def _norm(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values("employee_id").reset_index(drop=True)

def _case(rows, expected_rows):
    employees = pd.DataFrame(rows, columns=["employee_id", "name", "reports_to", "age"])
    employees["reports_to"] = employees["reports_to"].astype("Int64")
    got = sol_4(employees)
    exp = pd.DataFrame(expected_rows, columns=["employee_id", "name", "reports_count", "average_age"])
    assert_frame_equal(_norm(got), _norm(exp), check_dtype=False)

def run():
    cases = [
        ("example1", lambda: _case(
            [(9, "Hercy", None, 43),
             (6, "Alice", 9, 41),
             (4, "Bob", 9, 36),
             (2, "Winston", None, 37)],
            [(9, "Hercy", 2, 38)],
        )),
        ("example2", lambda: _case(
            [(1, "Michael", None, 45),
             (2, "Alice", 1, 38),
             (3, "Bob", 1, 42),
             (4, "Charlie", 2, 34),
             (5, "David", 2, 40),
             (6, "Eve", 3, 37),
             (7, "Frank", None, 50),
             (8, "Grace", None, 48)],
            [(1, "Michael", 2, 40),
             (2, "Alice", 2, 37),
             (3, "Bob", 1, 37)],
        )),
        ("no_managers", lambda: _case(
            [(1, "A", None, 20), (2, "B", None, 30)],
            [],
        )),
        ("one_manager_one_report", lambda: _case(
            [(1, "M", None, 50), (2, "R", 1, 30)],
            [(1, "M", 1, 30)],
        )),
        ("rounding_up", lambda: _case(
            [(1, "M", None, 50), (2, "R1", 1, 30), (3, "R2", 1, 31)],
            [(1, "M", 2, 30)],
        )),
        ("rounding_down", lambda: _case(
            [(1, "M", None, 50), (2, "R1", 1, 30), (3, "R2", 1, 30)],
            [(1, "M", 2, 30)],
        )),
        ("multiple_managers", lambda: _case(
            [(1, "M1", None, 40),
             (2, "E1", 1, 25),
             (3, "E2", 1, 35),
             (4, "M2", None, 55),
             (5, "E3", 4, 45),
             (6, "E4", 4, 46)],
            [(1, "M1", 2, 30), (4, "M2", 2, 46)],
        )),
    ]
    failed = 0
    for _, f in cases:
        try:
            f()
        except Exception:
            failed += 1
    print(f'Failed: {failed}')

run()